In [ ]:
import numpy as np
import random

from sklearn.metrics import precision_score, recall_score
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

In [ ]:
def compute_bootstrap_ci(num_iterations=10000, confidence_percentile=95):
    """
    Computes precision, recall, and F1-score confidence intervals using bootstrapping.

    Parameters:
    - num_iterations: Number of bootstrap iterations (default: 10000)
    - confidence_percentile: The percentile for confidence intervals (default: 95)

    Returns:
    - f1_final_array: The confidence interval values for F1-score
    """
    
    # Sort the class distribution and convert counts to percentages
    total_samples = 207+53+125+154
    class_counts = {
        'Cytosolic': 207,  # Cytosolic
        'Mitochondrial': 53,  # Mitochondrial
        'Nuclear': 125,  # Nuclear
        'Secretory': 154   # Secretory
    }

    class_distribution = [(label, round(count / total_samples * 10000)) for label, count in class_counts.items()]

    # Create a vector of 10,000 true labels based on the class distribution
    true_labels_vector = np.array([class_name for class_name, count in class_distribution for _ in range(count)])
    random.shuffle(true_labels_vector)

    # Sample predicted labels randomly from true_labels_vector
    pred_labels_vector = np.random.choice(true_labels_vector, size=len(true_labels_vector), replace=True)

    # Ensure both vectors have the same shape
    assert true_labels_vector.shape == pred_labels_vector.shape

    # Initialize arrays for storing precision, recall, and F1 scores
    precision_array = np.zeros((len(class_counts), num_iterations))
    recall_array = np.zeros((len(class_counts), num_iterations))
    f1_array = np.zeros((len(class_counts), num_iterations))

    # Perform bootstrapping by shuffling labels and computing metrics
    for i in range(num_iterations):
        # Shuffle the true and predicted labels independently
        shuffled_true = np.random.permutation(true_labels_vector)
        shuffled_pred = np.random.permutation(pred_labels_vector)

        # Compute precision and recall for each class
        precision = precision_score(shuffled_true, shuffled_pred, average=None, zero_division=0)
        recall = recall_score(shuffled_true, shuffled_pred, average=None, zero_division=0)

        # Compute F1-score, handling division by zero safely
        f1 = np.where((precision + recall) == 0, 0, 
                      2 * (precision * recall) / (precision + recall))

        # Store results in the respective arrays
        precision_array[:, i] = precision
        recall_array[:, i] = recall
        f1_array[:, i] = f1

    # Sort each row in the arrays from smallest to largest
    precision_array_sorted = np.sort(precision_array, axis=1)
    recall_array_sorted = np.sort(recall_array, axis=1)
    f1_array_sorted = np.sort(f1_array, axis=1)

    # Extract the confidence percentile value (e.g., 95th percentile)
    ci_index = int(num_iterations * (confidence_percentile / 100)) - 1  # Zero-based index

    precision_final_array = precision_array_sorted[:, ci_index]
    recall_final_array = recall_array_sorted[:, ci_index]
    f1_final_array = f1_array_sorted[:, ci_index]

    return precision_final_array, recall_final_array, f1_final_array, f1_array_sorted

In [ ]:
set_seed = 42
np.random.seed(set_seed)

prec_sorted, recall_sorted, f1_sorted, f1_array_sorted = compute_bootstrap_ci()

In [ ]:
labels = ['Cytosol', 'Mitochondria', 'Nucleus', 'Secretory']

plt.figure(figsize=(10, 6))
for i in range(4):
    density = gaussian_kde(f1_array_sorted[i])
    xs = np.linspace(f1_array_sorted[i].min(), f1_array_sorted[i].max(), 200)
    plt.plot(xs, density(xs), label=labels[i])
    f1_final = f1_array_sorted[:, -501]  # 95th percentile index
    plt.axvline(f1_final[i], color='black', linestyle='--', ymax=0.95)
    plt.plot(xs, density(xs), color='black')
plt.xlabel('F1-score')
plt.ylabel('Density')
plt.xlim(0, 0.5)
plt.ylim(0, 100)
plt.title('Bootstrap F1-score Distribution per Class')